In [21]:
import os
import os.path
import pandas as pd
import numpy as np

In [22]:
datadir = "data"
data = os.path.join(datadir, "adult.data")
df = pd.read_csv(data)
df.columns = ["age", "workclass", "fnlwgt", "education", "education-num", 
              "marital-status", "occupation", "relationship", "race", "sex", 
              "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]

In [ ]:
def avg_equiv_class_size_metric(data, qids, k):
    """
    Return the average sizes of the equivalence classes with respect to the QID set.
    Input:
        data: The input k-anonymized dataframe
        qids: the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return eq_classes.mean() / k


def discernability_metrics(data, qids):
    """
    Return the discernability score of the dataset with respect to the QID set.
    The discernability is calculated by assigning the penalty to each tuple depending 
    on the how many tuples are indistinguishable from it.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return (eq_classes ** 2).sum()


def classification_metrics(data, qids, sensitive_attr):
    """
    Return the classification metric score of the data with respect to the QID set,
    where we assign a penalty to each tuple t. If t's sensitive attribute matches 
    the majority sensitive attribute, the penalty = 0. Otherwise, penalty = size of the equivalence class.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    penalties = 0
    for _, group in data.groupby(list(qids)):
        # class_size = len(group)
        majority = group[sensitive_attr].value_counts().idxmax()
        mismatches = group[group[sensitive_attr] != majority]
        penalties += len(mismatches)
    return penalties / len(data)

In [ ]:

def mondrian(data, qids, k):
    """
    Recursive Mondrian-like anonymization.
    Splits dataset by widest QID until partitions cannot be split while keeping size >= k.
    """
    if len(data) < 2 * k:
        return [data]

    # Pick attribute with widest span
    spans = {}
    for q in qids:
        if pd.api.types.is_numeric_dtype(data[q]):
            spans[q] = data[q].max() - data[q].min()
        else:
            spans[q] = data[q].nunique()

    split_attr = max(spans, key=spans.get)

    # Split numeric vs categorical
    if pd.api.types.is_numeric_dtype(data[split_attr]):
        median = data[split_attr].median()
        left = data[data[split_attr] <= median]
        right = data[data[split_attr] > median]
    else:
        values = data[split_attr].value_counts().index.tolist()
        half = len(values) // 2
        left_vals, right_vals = values[:half], values[half:]
        left = data[data[split_attr].isin(left_vals)]
        right = data[data[split_attr].isin(right_vals)]

    # If invalid split, stop
    if len(left) < k or len(right) < k:
        return [data]

    return mondrian(left, qids, k) + mondrian(right, qids, k)


def generalize_partition(partition, qids):
    """
    Generalize values in a partition:
    - Numeric attributes become ranges
    - Categorical attributes become comma-separated sets
    """
    generalized = partition.copy()
    for q in qids:
        if pd.api.types.is_numeric_dtype(partition[q]):
            generalized[q] = f"{partition[q].min()}-{partition[q].max()}"
        else:
            generalized[q] = ",".join(sorted(partition[q].unique()))
    return generalized


def run_k_anonymity(data, qids, sensitive_attr, k, save_csv=True):
    """
    Run Mondrian anonymization and return:
      - The anonymized dataframe
      - Utility metrics (avg equivalence size, classification, discernability)
    """
    partitions = mondrian(data, qids, k)
    generalized_partitions = [generalize_partition(p, qids) for p in partitions]
    anon_df = pd.concat(generalized_partitions, ignore_index=True)

    # Compute utility metrics
    metrics = {
        "avg_eq_class_size": avg_equiv_class_size_metric(anon_df, qids, k),
        "classification_metric": classification_metrics(anon_df, qids, sensitive_attr),
        "discernability_metric": discernability_metrics(anon_df, qids),
    }

    if save_csv:
        anon_df.to_csv(f"adult_k{k}.csv", index=False)

    return anon_df, metrics

In [ ]:
QIDs = ["age", "workclass", "education", "marital-status",
        "occupation", "relationship", "race", "sex", "native-country"]
sensitive_attr = "income"

for k in [5, 10, 20]:
    anon_df, metrics = run_k_anonymity(df, QIDs, sensitive_attr, k)
    print(f"\nResults for k={k}")
    print(metrics)